In [ ]:
!pip install google-play-scraper pandas

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd


In [ ]:
result, continuation_token = reviews(
'com.starbucks.id',
lang='id',
country='id',
sort=Sort.NEWEST,
count=500
)

In [ ]:
data = pd.DataFrame(result)[['userName','content','score','at']]


In [ ]:
data.to_csv("dataset_review_starbucks.csv", index=False)
from google.colab import files
files.download("dataset_review_starbucks.csv")


**PROSES ANALISIS, BERIKUT INI**

In [ ]:
# 1. Import library pengolahan data dan matematika
import pandas as pd
import numpy as np
# 2. Import library visualisasi data
import matplotlib.pyplot as plt
import seaborn as sns
# 3. Import library machine learning dan data mining
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
# Konfirmasi bahwa semua library telah berhasil dimuat
print("Semua library berhasil dimuat dan siap digunakan.")

In [ ]:
# Mengimpor modul files dari library google.colab untuk unggah file
from google.colab import files
# Perintah untuk memunculkan tombol 'Choose Files'
uploaded = files.upload()

In [ ]:
# Memuat dataset dari file CSV ke dalam variabel 'df'
df = pd.read_csv('dataset_review_starbucks.csv')
# Menampilkan 5 data teratas untuk memastikan data termuat dengan benar
display(df.head())

In [ ]:
# Melihat dimensi dataset
print(f"Dimensi Dataset: {df.shape[0]} baris dan {df.shape[1]} kolom")
# Menampilkan informasi detail tipe data dan cek missing values
print("\n--- Informasi Struktur Data ---")
df.info()


In [ ]:
# Menampilkan ringkasan statistik untuk data numerik
display(df.describe())

In [ ]:
# Membuat subplot untuk dua visualisasi (Rating dan Panjang Ulasan)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# 1. Visualisasi Distribusi Rating (Bar Chart)
sns.countplot(x='score', data=df, palette='viridis', ax=ax[0])
ax[0].set_title('Distribusi Rating Ulasan')
ax[0].set_xlabel('Skor Rating (1-5)')
ax[0].set_ylabel('Jumlah Ulasan')
# 2. Visualisasi Panjang Ulasan (Histogram)
df['length'] = df['content'].apply(len)
sns.histplot(df['length'], bins=30, kde=True, color='skyblue', ax=ax[1])
ax[1].set_title('Distribusi Panjang Ulasan (Karakter)')
ax[1].set_xlabel('Jumlah Karakter')
ax[1].set_ylabel('Frekuensi')
plt.tight_layout()
plt.show()

In [ ]:
import re
# Fungsi untuk membersihkan teks (case folding dan menghapus simbol/angka)
def clean_text(text):
 # Mengubah teks menjadi huruf kecil
 text = text.lower()
 # Menghapus karakter non-alfabet (simbol, angka, dll)
 text = re.sub(r'[^a-z\s]', '', text)
 return text
# Menerapkan fungsi pada seluruh data (seluruh 500 ulasan)
df['content_cleaned'] = df['content'].apply(clean_text)
# Verifikasi jumlah data yang telah dibersihkan
print(f"Total data yang telah dibersihkan: {len(df['content_cleaned'])} data")
# Menampilkan 5 data teratas untuk pengecekan
display(df[['content', 'content_cleaned']].head())

In [ ]:
# Instalasi library Sastrawi jika belum ada
!pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import
StopWordRemoverFactory
# Inisialisasi Stopword Remover
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()
# Menghapus stopword dari setiap ulasan
df['content_cleaned'] = df['content_cleaned'].apply(lambda x:
stopword_remover.remove(x))

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
import warnings
# Mengabaikan peringatan agar tampilan output tetap bersih
warnings.filterwarnings("ignore")
# Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()
# Membuat kamus (dictionary) untuk menyimpan hasil stemming agar tidak duplikat
stemmed_cache = {}
# Fungsi stemming dengan optimasi cache
def stem_fast_tqdm(text):
 words = str(text).split()
 stemmed_words = []
 for word in words:
 # Jika kata belum pernah di-stemming, maka proses
 if word not in stemmed_cache:
 stemmed_cache[word] = stemmer.stem(word)
 # Ambil hasil dari kamus
 stemmed_words.append(stemmed_cache[word])
 return ' '.join(stemmed_words)
# Mengaktifkan progress bar untuk pandas
tqdm.pandas()
# Melakukan stemming pada teks yang telah dibersihkan
df['content_cleaned'] = df['content_cleaned'].progress_apply(stem_fast_tqdm)
# Menampilkan 5 data teratas setelah proses stemming
display(df[['content', 'content_cleaned']].head())

In [ ]:
# Cek jumlah baris yang ada di kolom content_cleaned
print(f"Jumlah total baris yang diproses: {len(df['content_cleaned'])}")
# Cek 5 data terbawah (untuk memastikan bagian akhir juga sudah diproses)
display(df[['content', 'content_cleaned']].tail())

In [ ]:
# Fungsi untuk pelabelan
def label_sentiment(score):
 if score >= 4:
 return 'Positif'
 elif score == 3:
 return 'Netral'
 else:
 return 'Negatif'
# Menerapkan label
df['sentiment'] = df['score'].apply(label_sentiment)
display(df[['score', 'sentiment']].head())

In [ ]:
from sklearn.model_selection import train_test_split
X = df['content_cleaned']
y = df['sentiment']
# Pembagian data 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Jumlah Data Latih: {len(X_train)}, Jumlah Data Uji: {len(X_test)}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
# Membuat pipeline untuk menggabungkan Vektorizer (TF-IDF) dan Klasifier (Naive
Bayes)
# Pipeline memastikan proses vektorisasi dan klasifikasi berjalan dalam satu alur
model = make_pipeline(TfidfVectorizer(), MultinomialNB())
# Melatih model menggunakan data latih (X_train dan y_train)
model.fit(X_train, y_train)
# Melakukan prediksi menggunakan data uji
y_pred = model.predict(X_test)
print("Proses pelatihan model Naive Bayes telah selesai.")

In [ ]:
# Kode Visualisasi 1
plt.figure(figsize=(6, 4))
sns.countplot(x='sentiment', data=df, palette='viridis')
plt.title('Distribusi Kelas Sentimen dalam Dataset')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah Ulasan')
plt.show()

In [ ]:
# 1. Mengubah Classification Report menjadi DataFrame agar mudah di-plot
report = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()
# 2. Menghapus baris rata-rata agar hanya fokus pada tiap kategori sentimen
df_report = df_report.drop(['accuracy', 'macro avg', 'weighted avg'])
# 3. Visualisasi Bar Plot
df_report[['precision', 'recall', 'f1-score']].plot(kind='bar', figsize=(10, 6))
plt.title('Performa Model per Kelas Sentimen')
plt.xlabel('Sentimen')
plt.ylabel('Skor')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Inisialisasi TF-IDF Vectorizer
# max_features=1000 berarti kita hanya mengambil 1000 kata paling berpengaruh
vectorizer = TfidfVectorizer(max_features=1000)
# Mengubah data teks menjadi matriks numerik (TF-IDF)
tfidf_matrix = vectorizer.fit_transform(df['content_cleaned'])
# Menampilkan dimensi matriks hasil seleksi fitur
print(f"Dimensi matriks TF-IDF: {tfidf_matrix.shape}")

In [ ]:
# Inisialisasi model K-Means dengan k=3
# n_init=10 memastikan algoritma berjalan dengan inisialisasi pusat klaster yang stabil
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
# Melakukan klasterisasi pada matriks TF-IDF
df['cluster'] = kmeans.fit_predict(tfidf_matrix)
# Menampilkan 5 data teratas dengan label klasternya
display(df[['content', 'cluster']].head())

In [ ]:
# Menampilkan 3 sampel ulasan untuk setiap klaster
for i in range(3):
 print(f"\n--- Sampel Ulasan Klaster {i} ---")
 # Mengambil sampel ulasan dari klaster i
 display(df[df['cluster'] == i][['content']].head(3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualisasi jumlah ulasan per klaster
plt.figure(figsize=(6, 4))
sns.countplot(x='cluster', data=df, palette='viridis')
plt.title('Distribusi Jumlah Ulasan per Klaster')
plt.xlabel('Klaster')
plt.ylabel('Jumlah Ulasan')
plt.show()

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
# 1. Mengubah teks menjadi format list of words (transaksi)
dataset_transaction = df['content_cleaned'].apply(lambda x: x.split()).tolist()
# 2. One-Hot Encoding (Ubah ke tabel biner)
te = TransactionEncoder()
te_ary = te.fit(dataset_transaction).transform(dataset_transaction)
df_apriori = pd.DataFrame(te_ary, columns=te.columns_)
# 3. Mencari aturan asosiasi
frequent_itemsets = apriori(df_apriori, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.2)
rules = rules.sort_values(by='lift', ascending=False)

In [ ]:
# Menampilkan tabel hasil aturan asosiasi
print("Aturan Asosiasi (Hasil Analisis):")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Menyiapkan label aturan untuk grafik
top_rules = rules.head(10).copy()
top_rules['rule'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x))) + ' -> ' + \
 top_rules['consequents'].apply(lambda x: ', '.join(list(x)))
# Visualisasi Bar Chart Horizontal
plt.figure(figsize=(10, 6))
sns.barplot(x='support', y='rule', data=top_rules, palette='viridis')
plt.title('Top 10 Aturan Asosiasi Berdasarkan Support')
plt.xlabel('Support')
plt.ylabel('Aturan Asosiasi')
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
# Membuat subplot untuk 3 kategori sentimen
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sentiments = ['Positif', 'Netral', 'Negatif']
for i, sentiment in enumerate(sentiments):
 # Filter data berdasarkan sentimen
 text = ' '.join(df[df['sentiment'] == sentiment]['content_cleaned'])

 # Generate WordCloud
 wc = WordCloud(width=800, height=400, background_color='white').generate(text)

 # Plotting
 axes[i].imshow(wc, interpolation='bilinear')
 axes[i].set_title(f'WordCloud Sentimen {sentiment}')
 axes[i].axis('off')
plt.show()

In [ ]:
# Mengubah kolom 'at' menjadi format datetime
# Pastikan nama kolom 'at' sesuai dengan dataset kamu
df['at'] = pd.to_datetime(df['at'])
# Mengelompokkan data berdasarkan bulan dan menghitung jumlah ulasan
df_trend = df.set_index('at').resample('M').size()
# Visualisasi Line Chart
plt.figure(figsize=(10, 5))
df_trend.plot(kind='line', marker='o', color='red', linewidth=2)
plt.title('Tren Jumlah Ulasan Pengguna per Bulan')
plt.xlabel('Bulan')
plt.ylabel('Jumlah Ulasan')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()